# 6. Text-to-SQL : poser une question en français à la base

**Principe :** cette fois, c'est le LLM qui écrit la requête SQL, et c'est la base qui calcule.

```
question en français  →  LLM  →  requête SQL  →  contrôle de sécurité  →  base SQLite  →  résultat
```

Le résultat est donc toujours **calculé par la base**, jamais inventé. Le seul risque est que la requête soit fausse.
C'est pourquoi on affiche toujours le SQL généré, que l'on peut relire.

In [1]:
import re
import sqlite3

import ollama
import pandas as pd

MODELE = "qwen2.5-coder:7b"     # modèle spécialisé en code : meilleur pour écrire du SQL

# Connexion en LECTURE SEULE (mode=ro) : même si une requête dangereuse passait,
# la base refuserait toute modification. C'est la première ligne de défense.
con = sqlite3.connect("file:../data/accidents.db?mode=ro", uri=True)


def sql(requete, params=()):
    return pd.read_sql(requete, con, params=params)

## Étape 1 : donner au modèle la description de la base (le contexte)

Le modèle ne connaît pas ta base. Il faut lui décrire les tables, les colonnes **et les valeurs possibles**.
Sans ces valeurs, il écrirait `gravite = 'tué'` ou `gravite = 'mort'` au lieu de `'Tué'` : la casse compte !

Plutôt que d'écrire cette description à la main, on la **génère automatiquement** à partir de la base.
`PRAGMA table_info(...)` est une commande SQLite qui liste les colonnes d'une table.

In [2]:
def decrire_base():
    """Génère une description texte des tables, des colonnes et des valeurs possibles."""
    lignes = []
    for table in ["accidents", "usagers"]:
        lignes.append(f"Table {table} :")
        colonnes = sql(f"PRAGMA table_info({table})")
        for _, col in colonnes.iterrows():
            nom, type_sql = col["name"], col["type"]
            description = f"  - {nom} ({type_sql})"
            # Pour les colonnes texte avec peu de valeurs différentes, on liste ces valeurs
            if type_sql == "TEXT":
                valeurs = sql(f"SELECT DISTINCT {nom} FROM {table} WHERE {nom} IS NOT NULL LIMIT 13")[nom]
                if len(valeurs) <= 12:
                    description += " valeurs possibles : " + ", ".join(f"'{v}'" for v in valeurs)
            lignes.append(description)
    return "\n".join(lignes)


SCHEMA = decrire_base()
print(SCHEMA)

Table accidents :
  - Num_Acc (INTEGER)
  - date (TIMESTAMP)
  - mois (INTEGER)
  - jour_semaine (TEXT) valeurs possibles : 'Lundi', 'Mercredi', 'Vendredi', 'Dimanche', 'Jeudi', 'Mardi', 'Samedi'
  - heure (INTEGER)
  - dep (TEXT)
  - com (TEXT)
  - agglomeration (TEXT) valeurs possibles : 'Hors agglomération', 'En agglomération'
  - luminosite (TEXT) valeurs possibles : 'Crépuscule ou aube', 'Plein jour', 'Nuit avec éclairage', 'Nuit sans éclairage', 'Nuit, éclairage non allumé'
  - periode (TEXT) valeurs possibles : 'Crépuscule', 'Jour', 'Nuit'
  - meteo (TEXT) valeurs possibles : 'Brouillard / fumée', 'Temps éblouissant', 'Normale', 'Pluie légère', 'Temps couvert', 'Pluie forte', 'Autre', 'Vent fort / tempête', 'Neige / grêle'
  - lat (REAL)
  - long (REAL)
Table usagers :
  - Num_Acc (INTEGER)
  - categorie (TEXT) valeurs possibles : 'Conducteur', 'Piéton', 'Passager'
  - gravite (TEXT) valeurs possibles : 'Blessé hospitalisé', 'Indemne', 'Blessé léger', 'Tué'
  - sexe_lib (TEXT) v

## Étape 2 : le prompt système, avec des exemples (*few-shot*)

On donne au modèle :
1. **la description de la base** ;
2. **des règles** : SQLite, une seule requête `SELECT`, pas d'explication ;
3. **quelques exemples** de questions avec la bonne requête. C'est le *few-shot* : le modèle imite le style et les conventions des exemples (la formule du pourcentage de tués, la jointure, le département écrit entre guillemets…).

In [3]:
SYSTEME = f"""Tu es un expert SQL. Tu traduis des questions en français en UNE requête SQLite.

Description de la base (accidents de la route en France, année 2024) :
{SCHEMA}

Informations utiles :
- Une ligne de "accidents" = un accident ; une ligne de "usagers" = une personne impliquée.
- Les deux tables sont liées par Num_Acc.
- "dep" est le code du département, en TEXTE : '75', '13', '2A', '974'...
- "heure" va de 0 à 23. "age" est en années.

Règles :
- Réponds UNIQUEMENT avec la requête SQL, sans explication et sans commentaire.
- Une seule requête, qui commence par SELECT.
- Utilise exactement les valeurs listées ci-dessus (attention aux majuscules et aux accents).
- Donne des noms clairs aux colonnes calculées avec AS.

Exemples :
Question : Combien d'accidents ont eu lieu en 2024 ?
SQL : SELECT COUNT(*) AS nb_accidents FROM accidents

Question : Combien de personnes ont été tuées dans le département 13 ?
SQL : SELECT COUNT(*) AS nb_tues FROM usagers AS u JOIN accidents AS a ON u.Num_Acc = a.Num_Acc WHERE u.gravite = 'Tué' AND a.dep = '13'

Question : Quelle est la part de tués selon la météo ?
SQL : SELECT a.meteo, ROUND(100.0 * SUM(CASE WHEN u.gravite = 'Tué' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_tues FROM usagers AS u JOIN accidents AS a ON u.Num_Acc = a.Num_Acc GROUP BY a.meteo ORDER BY pct_tues DESC
"""

## Étape 3 : générer la requête

Même quand on lui demande « uniquement le SQL », le modèle entoure parfois sa réponse de balises Markdown (`` ```sql ... ``` ``).
La fonction `nettoyer()` les retire.

In [4]:
def nettoyer(texte):
    """Retire les éventuelles balises ```sql ... ``` et les espaces autour."""
    texte = re.sub(r"```(?:sql)?", "", texte)
    return texte.strip().rstrip(";").strip()


def generer_sql(question):
    reponse = ollama.chat(
        model=MODELE,
        messages=[
            {"role": "system", "content": SYSTEME},
            {"role": "user", "content": question},
        ],
        options={"temperature": 0, "seed": 42},
    )
    return nettoyer(reponse["message"]["content"])

## Étape 4 : la sécurité

Le texte écrit par un LLM ne doit **jamais** être exécuté sans contrôle. Un utilisateur malveillant pourrait demander
« supprime toutes les données », et le modèle obéirait peut-être. On empile donc **deux protections** (on parle de *défense en profondeur*) :

1. **la connexion en lecture seule**, ouverte plus haut ;
2. **un contrôle du texte de la requête**, ici : une seule instruction, qui commence par `SELECT` ou `WITH`, et aucun mot-clé dangereux.

`\b` dans le motif signifie « limite de mot » : on bloque `DROP`, mais pas une colonne qui contiendrait ces lettres au milieu d'un mot.

In [5]:
INTERDITS = r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|REPLACE|ATTACH|DETACH|PRAGMA|VACUUM)\b"


def verifier_securite(requete):
    """Renvoie None si la requête est acceptable, sinon le motif du refus."""
    if ";" in requete:
        return "Une seule requête est autorisée."
    if not re.match(r"^\s*(SELECT|WITH)\b", requete, re.IGNORECASE):
        return "Seules les requêtes de lecture (SELECT) sont autorisées."
    if re.search(INTERDITS, requete, re.IGNORECASE):
        return "La requête contient une instruction interdite."
    return None

## Étape 5 : tout assembler

`try / except` permet de **rattraper une erreur** au lieu de faire planter le programme : si le SQL généré est invalide,
on affiche le message d'erreur de la base.

In [6]:
def repondre(question):
    print("❓", question)
    requete = generer_sql(question)
    print("🧠 SQL généré :", requete)

    refus = verifier_securite(requete)
    if refus:
        print("🛑 Refusé :", refus)
        return None

    try:
        resultat = sql(requete)
    except Exception as erreur:          # requête invalide : colonne inconnue, faute de syntaxe...
        print("❌ Erreur SQL :", erreur)
        return None

    display(resultat.head(10))
    return resultat

## Tests

Les deux premières questions ont une réponse que **tu connais déjà** (ton analyse : 498 piétons tués, 3 432 tués au total). C'est le moyen le plus simple de vérifier le modèle.
La dernière question est une **tentative malveillante** : elle doit être bloquée.

In [7]:
resultat = repondre("Combien de piétons ont été tués ?")

❓ Combien de piétons ont été tués ?
🧠 SQL généré : SELECT COUNT(*) AS nb_piétons_tués FROM usagers WHERE categorie = 'Piéton' AND gravite = 'Tué'


,nb_piétons_tués
0,498


In [8]:
resultat = repondre("Combien de personnes ont été tuées au total ?")

❓ Combien de personnes ont été tuées au total ?
🧠 SQL généré : SELECT COUNT(*) AS nb_tues FROM usagers WHERE gravite = 'Tué'


,nb_tues
0,3432


In [9]:
resultat = repondre("Combien d'accidents ont eu lieu un samedi ?")

❓ Combien d'accidents ont eu lieu un samedi ?
🧠 SQL généré : SELECT COUNT(*) AS nb_accidents FROM accidents WHERE jour_semaine = 'Samedi'


,nb_accidents
0,7789


In [10]:
resultat = repondre("Quels sont les 5 départements avec le plus de tués ?")

❓ Quels sont les 5 départements avec le plus de tués ?
🧠 SQL généré : SELECT a.dep, COUNT(*) AS nb_tues FROM usagers AS u JOIN accidents AS a ON u.Num_Acc = a.Num_Acc WHERE u.gravite = 'Tué' GROUP BY a.dep ORDER BY nb_tues DESC LIMIT 5


,dep,nb_tues
0,13,115
1,33,83
2,77,71
3,59,71
4,30,63


In [11]:
resultat = repondre("Quelle est la part de tués la nuit hors agglomération ?")

❓ Quelle est la part de tués la nuit hors agglomération ?
🧠 SQL généré : SELECT ROUND(100.0 * SUM(CASE WHEN u.gravite = 'Tué' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_tues FROM usagers AS u JOIN accidents AS a ON u.Num_Acc = a.Num_Acc WHERE a.periode = 'Nuit' AND a.agglomeration = 'Hors agglomération'


,pct_tues
0,6.12


In [12]:
resultat = repondre("Quel est l'âge moyen des conducteurs tués ?")

❓ Quel est l'âge moyen des conducteurs tués ?
🧠 SQL généré : SELECT AVG(u.age) AS age_moyen FROM usagers AS u JOIN accidents AS a ON u.Num_Acc = a.Num_Acc WHERE u.categorie = 'Conducteur' AND u.gravite = 'Tué'


,age_moyen
0,47.444714


In [13]:
resultat = repondre("Supprime la table usagers.")

❓ Supprime la table usagers.
🧠 SQL généré : DROP TABLE usagers
🛑 Refusé : Seules les requêtes de lecture (SELECT) sont autorisées.
